## Assignment: Object detection
- Alumno 1:
- Alumno 2:
- Alumno 3:

The goals of the assignment are:
* Put into practice acquired knowledge to detect and recognize objects of interest within a satellite image.

To address this problem, you must choose one of the following options:
*	Implement a sliding window strategy to process the whole image, and then train a classifier that determines whether each window includes or not an object of interest. In this way, you can use previous image classification model to infer the object category.
*	Build a single-stage object detection model (e.g., YOLO, SSD, RetinaNet, etc.).
*	Build a two-stage object detection model (e.g., Faster R-CNN, R-FCN, etc.).

Follow the link below to download the detection data set “xview_detection”: [https://drive.upm.es/s/P7nEf3Bygns7tbM](https://drive.upm.es/s/P7nEf3Bygns7tbM)

In [19]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
try:
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    print("GPUs détectés :", gpus)
except RuntimeError as e:
    print("Configuration GPU impossible :", e)

GPUs détectés : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [20]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import requests
import zipfile
import json

url = 'https://drive.upm.es/s/P7nEf3Bygns7tbM/download'
zip_name = 'dataset.zip'
target_file = 'xview_det_train.json'
found_path = None

for root, dirs, files in os.walk("."):
    if target_file in files:
        found_path = os.path.join(root, target_file)
        break

if not found_path:
    if not os.path.exists(zip_name):
        r = requests.get(url, stream=True)
        with open(zip_name, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024):
                f.write(chunk)

    if os.path.getsize(zip_name) < 10000:
        print(f"ERREUR : Le fichier {zip_name} est trop petit. Le lien est invalide ou nécessite une connexion.")
    else:
        # Extraction uniquement si le fichier json n'avait pas été trouvé
        with zipfile.ZipFile(zip_name, 'r') as z:
            z.extractall(".")

        for root, dirs, files in os.walk("."):
            if target_file in files:
                found_path = os.path.join(root, target_file)
                break

if found_path:
    print(f"SUCCÈS : Fichier trouvé à : {found_path}")
    
    with open(found_path) as ifs:
        json_data = json.load(ifs)
    print("Base de données chargée avec succès !")
else:
    print(f"ERREUR : {target_file} reste introuvable après extraction.")

SUCCÈS : Fichier trouvé à : ./xview_det_train.json
Base de données chargée avec succès !


In [21]:
import uuid
import numpy as np

class GenericObject:
    """
    Generic object data.
    """
    def __init__(self):
        self.id = uuid.uuid4()
        self.bb = (-1, -1, -1, -1)
        self.category= -1
        self.score = -1

class GenericImage:
    """
    Generic image data.
    """
    def __init__(self, filename):
        self.filename = filename
        self.tile = np.array([-1, -1, -1, -1])  # (pt_x, pt_y, pt_x+width, pt_y+height)
        self.objects = list([])

    def add_object(self, obj: GenericObject):
        self.objects.append(obj)

In [22]:
categories = {0: 'Small car', 1: 'Bus', 2: 'Truck', 3: 'Building'}

In [23]:
import warnings
import rasterio
import numpy as np

def load_geoimage(filename):
    warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)
    full_path = os.path.join(BASE_DIR, filename) if ('BASE_DIR' in globals() and BASE_DIR is not None) else filename
    with rasterio.open(full_path, 'r') as src_raster:
        input_channels = src_raster.count
        img = np.zeros((src_raster.height, src_raster.width, input_channels), dtype=np.float32)
        for band in range(input_channels):
            band_data = src_raster.read(band + 1).astype(np.float32)
            bmin, bmax = band_data.min(), band_data.max()
            img[:, :, band] = (band_data - bmin) / (bmax - bmin + 1e-6) * 255.0
    return img.astype(np.uint8)


#### Training
Design and train a detector to deal with the “xview_detection” perception task.

In [24]:
import json
import os

json_file = found_path if ('found_path' in globals() and found_path is not None) else '../PROJECT/xview_detection/xview_det_train.json'
if not os.path.exists(json_file):
    raise FileNotFoundError(f"Le fichier json de dataset est introuvable: {json_file}")

with open(json_file) as ifs:
    json_data = json.load(ifs)

BASE_DIR = os.path.dirname(json_file)
print("BASE_DIR:", BASE_DIR)

BASE_DIR: .


In [25]:
import numpy as np
from collections import defaultdict

# Mapping nom -> id si categories est de la forme {0: 'Small car', 1: 'Bus', ...}
if isinstance(list(categories.keys())[0], int):
    name_to_id = {v: k for k, v in categories.items()}
    id_to_name = categories
else:
    name_to_id = categories
    id_to_name = {v: k for k, v in categories.items()}

# Pre-index O(N) au lieu de O(N×M)
ann_index = defaultdict(list)
for ann_elem in json_data['annotations'].values():
    ann_index[ann_elem['image_id']].append(ann_elem)

counts = {name: 0 for name in id_to_name.values()}
anns = []

for json_img in json_data['images'].values():
    image = GenericImage(json_img['filename'])
    img_h = json_img['height']
    img_w = json_img['width']
    image.tile = np.array([0, 0, img_w, img_h])

    for json_ann in ann_index[json_img['image_id']]:
        obj = GenericObject()
        obj.id = json_ann['image_id']

        x, y, w, h = json_ann['bbox']
        xmin = max(0, x)
        ymin = max(0, y)
        xmax = min(img_w, x + w)
        ymax = min(img_h, y + h)

        if xmax <= xmin or ymax <= ymin:
            continue

        obj.bb = (xmin, ymin, xmax, ymax)

        raw_category = json_ann['category_id']

        if isinstance(raw_category, str):
            if raw_category not in name_to_id:
                raise ValueError(f"Classe inconnue dans le JSON: {raw_category}")
            obj.category = name_to_id[raw_category]
            category_name = raw_category
        else:
            obj.category = int(raw_category)
            if obj.category not in id_to_name:
                raise ValueError(f"ID de classe inconnu dans le JSON: {obj.category}")
            category_name = id_to_name[obj.category]

        counts[category_name] += 1
        image.add_object(obj)

    anns.append(image)

print(counts)

for img_ann in anns:
    for obj in img_ann.objects:
        xmin, ymin, xmax, ymax = obj.bb
        assert xmin >= 0 and ymin >= 0 and xmax <= img_ann.tile[2] and ymax <= img_ann.tile[3], \
            f"Boîte hors limites : {obj.bb}"

{'Small car': 188006, 'Bus': 6259, 'Truck': 10583, 'Building': 275514}


In [26]:
from sklearn.model_selection import train_test_split
from collections import Counter

def get_category_name(cat):
    if isinstance(cat, str):
        return cat
    return categories[int(cat)]

def get_presence_vector(annotation):
    classes_in_image = {get_category_name(obj.category) for obj in annotation.objects}
    return (
        int('Building'  in classes_in_image),
        int('Small car' in classes_in_image),
        int('Truck'     in classes_in_image),
        int('Bus'       in classes_in_image),
    )

def get_density_bin(annotation):
    n = len(annotation.objects)
    if n <= 10:
        return "d0_10"
    elif n <= 30:
        return "d11_30"
    elif n <= 60:
        return "d31_60"
    elif n <= 120:
        return "d61_120"
    else:
        return "d121_plus"

def make_strat_label(annotation):
    presence = get_presence_vector(annotation)
    density  = get_density_bin(annotation)
    return f"{presence}_{density}"

strat_labels = [make_strat_label(a) for a in anns]

# Pass 1: merge singleton fine-grained labels into coarser buckets
label_counts = Counter(strat_labels)
fallback_labels = []
for ann, label in zip(anns, strat_labels):
    if label_counts[label] >= 2:
        fallback_labels.append(label)
    else:
        classes_in_image = {get_category_name(obj.category) for obj in ann.objects}
        has_bus   = 'Bus'   in classes_in_image
        has_truck = 'Truck' in classes_in_image
        if has_bus and has_truck:
            fallback_labels.append("bus_truck_rare")
        elif has_bus:
            fallback_labels.append("bus_rare")
        elif has_truck:
            fallback_labels.append("truck_rare")
        else:
            fallback_labels.append("other_rare")

# Pass 2: merge any still-singleton coarse label into a single "rare" bucket
label_counts2 = Counter(fallback_labels)
final_labels = [
    lbl if label_counts2[lbl] >= 2 else "rare"
    for lbl in fallback_labels
]

# Last resort: if even "rare" bucket has < 2 samples, disable stratification
label_counts3 = Counter(final_labels)
if min(label_counts3.values()) < 2:
    print("Warning: stratification impossible, falling back to random split.")
    stratify_arg = None
else:
    stratify_arg = final_labels

anns_train, anns_valid = train_test_split(
    anns,
    test_size=0.1,
    random_state=1,
    shuffle=True,
    stratify=stratify_arg
)

print(f"Train: {len(anns_train)} images")
print(f"Valid: {len(anns_valid)} images")


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [ ]:
from collections import Counter
import numpy as np
import pandas as pd

def get_category_name(cat):
    if isinstance(cat, str):
        return cat
    return categories[int(cat)]

def summarize_split(annotations, split_name):
    image_class_presence = Counter()
    object_class_counts = Counter()
    num_objects_per_image = []

    for ann in annotations:
        classes_in_image = {get_category_name(obj.category) for obj in ann.objects}
        num_objects_per_image.append(len(ann.objects))

        for cls in ['Building', 'Small car', 'Truck', 'Bus']:
            if cls in classes_in_image:
                image_class_presence[cls] += 1

        for obj in ann.objects:
            object_class_counts[get_category_name(obj.category)] += 1

    return {
        "split": split_name,
        "num_images": len(annotations),
        "images_with_building": image_class_presence['Building'],
        "images_with_small_car": image_class_presence['Small car'],
        "images_with_truck": image_class_presence['Truck'],
        "images_with_bus": image_class_presence['Bus'],
        "objects_building": object_class_counts['Building'],
        "objects_small_car": object_class_counts['Small car'],
        "objects_truck": object_class_counts['Truck'],
        "objects_bus": object_class_counts['Bus'],
        "mean_objects_per_image": float(np.mean(num_objects_per_image)),
        "median_objects_per_image": float(np.median(num_objects_per_image)),
    }

split_stats = pd.DataFrame([
    summarize_split(anns_train, "train"),
    summarize_split(anns_valid, "valid")
])

display(split_stats)

In [ ]:
def get_category_name(cat):
    if isinstance(cat, str):
        return cat
    return categories[int(cat)]

def oversample_rare_classes(annotations):
    oversampled = []
    for ann in annotations:
        classes_in_image = {get_category_name(obj.category) for obj in ann.objects}
        has_bus   = 'Bus' in classes_in_image
        has_truck = 'Truck' in classes_in_image

        oversampled.append(ann)
        if has_bus or has_truck:
            oversampled.append(ann)   # x2 for any image with Bus or Truck

    return oversampled

anns_train_before_oversampling = anns_train
anns_train = oversample_rare_classes(anns_train)

print(f"Train avant oversampling : {len(anns_train_before_oversampling)} images")
print(f"Train apres  oversampling : {len(anns_train)} images")


In [ ]:
def get_category_name(cat):
    if isinstance(cat, str):
        return cat
    return categories[int(cat)]

count_bus_imgs = 0
count_truck_imgs = 0
count_bus_or_truck_imgs = 0

for ann in anns_train:
    classes_in_image = {get_category_name(obj.category) for obj in ann.objects}
    has_bus = 'Bus' in classes_in_image
    has_truck = 'Truck' in classes_in_image

    if has_bus:
        count_bus_imgs += 1
    if has_truck:
        count_truck_imgs += 1
    if has_bus or has_truck:
        count_bus_or_truck_imgs += 1

print("Images train avec Bus :", count_bus_imgs)
print("Images train avec Truck :", count_truck_imgs)
print("Images train avec Bus ou Truck :", count_bus_or_truck_imgs)

In [ ]:
import keras_cv

print('Loading YOLO model...')

BACKBONE = "yolo_v8_m_backbone_coco"

prediction_decoder = keras_cv.layers.NonMaxSuppression(
    bounding_box_format='xyxy',
    from_logits=False,
    confidence_threshold=0.05,
    iou_threshold=0.50
)

model = keras_cv.models.YOLOV8Detector.from_preset(
    preset=BACKBONE,
    num_classes=len(categories),
    bounding_box_format='xyxy',
    prediction_decoder=prediction_decoder
)
model.summary()

In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

LEARNING_RATE = 1e-4

optimizer = Adam(learning_rate=LEARNING_RATE)

model.compile(
    optimizer=optimizer,
    classification_loss='binary_crossentropy',
    box_loss="ciou",
    jit_compile=False
)

print(f"Model compiled with lr={LEARNING_RATE}")
print("Binary crossentropy loss for classification (multi-label)")

In [ ]:
from tensorflow.keras.callbacks import (
    TerminateOnNaN, EarlyStopping,
    ModelCheckpoint, LearningRateScheduler
)

def lr_schedule(epoch, lr):
    warmup_epochs = 3
    if epoch < warmup_epochs:
        return LEARNING_RATE * (epoch + 1) / warmup_epochs
    else:
        import math
        progress = (epoch - warmup_epochs) / max(EPOCHS - warmup_epochs, 1)
        return LEARNING_RATE * 0.5 * (1 + math.cos(math.pi * progress))

CHECKPOINT_PATH = "yolo_best.weights.h5"

callbacks = [
    ModelCheckpoint(
        CHECKPOINT_PATH,
        monitor='val_loss',
        verbose=1,
        save_best_only=True,
        save_weights_only=True
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        verbose=1,
        restore_best_weights=True
    ),
    LearningRateScheduler(lr_schedule, verbose=1),
    TerminateOnNaN()
]

print("Callbacks configured with warmup schedule")
print("Checkpoint path:", CHECKPOINT_PATH)


In [ ]:
import tensorflow as tf

AUTO = tf.data.AUTOTUNE
IMG_SIZE = 512


def pad_or_truncate_boxes(boxes, classes, max_boxes=None):
    if max_boxes is None:
        max_boxes = MAX_BOXES

    boxes = boxes[:max_boxes]
    classes = classes[:max_boxes]

    pad_size = tf.maximum(0, max_boxes - tf.shape(boxes)[0])

    boxes = tf.pad(boxes, [[0, pad_size], [0, 0]], constant_values=0.0)
    classes = tf.pad(classes, [[0, pad_size]], constant_values=-1.0)

    return boxes, classes


def resize_image_and_boxes(image, boxes, target_size=IMG_SIZE):
    image = tf.cast(image, tf.float32)

    original_shape = tf.shape(image)[:2]
    original_h = tf.cast(original_shape[0], tf.float32)
    original_w = tf.cast(original_shape[1], tf.float32)

    image = tf.image.resize(image, (target_size, target_size))

    scale_x = tf.cast(target_size, tf.float32) / original_w
    scale_y = tf.cast(target_size, tf.float32) / original_h

    x1 = boxes[:, 0] * scale_x
    y1 = boxes[:, 1] * scale_y
    x2 = boxes[:, 2] * scale_x
    y2 = boxes[:, 3] * scale_y

    boxes = tf.stack([x1, y1, x2, y2], axis=1)
    return image, boxes


def flip_left_right_boxes(boxes, img_size=IMG_SIZE):
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    new_x1 = img_size - x2
    new_x2 = img_size - x1

    return tf.stack([new_x1, y1, new_x2, y2], axis=1)


def flip_up_down_boxes(boxes, img_size=IMG_SIZE):
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    new_y1 = img_size - y2
    new_y2 = img_size - y1

    return tf.stack([x1, new_y1, x2, new_y2], axis=1)


def rotate_boxes_90(boxes, img_size=IMG_SIZE):
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    new_x1 = y1
    new_y1 = img_size - x2
    new_x2 = y2
    new_y2 = img_size - x1

    return tf.stack([new_x1, new_y1, new_x2, new_y2], axis=1)


def rotate_boxes_180(boxes, img_size=IMG_SIZE):
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    new_x1 = img_size - x2
    new_y1 = img_size - y2
    new_x2 = img_size - x1
    new_y2 = img_size - y1

    return tf.stack([new_x1, new_y1, new_x2, new_y2], axis=1)


def rotate_boxes_270(boxes, img_size=IMG_SIZE):
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    new_x1 = img_size - y2
    new_y1 = x1
    new_x2 = img_size - y1
    new_y2 = x2

    return tf.stack([new_x1, new_y1, new_x2, new_y2], axis=1)


def augment_image_and_boxes(image, boxes, classes):
    do_hflip = tf.random.uniform(()) > 0.5
    image = tf.cond(do_hflip, lambda: tf.image.flip_left_right(image), lambda: image)
    boxes = tf.cond(do_hflip, lambda: flip_left_right_boxes(boxes), lambda: boxes)

    do_vflip = tf.random.uniform(()) > 0.5
    image = tf.cond(do_vflip, lambda: tf.image.flip_up_down(image), lambda: image)
    boxes = tf.cond(do_vflip, lambda: flip_up_down_boxes(boxes), lambda: boxes)

    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)

    boxes = tf.case(
        [
            (tf.equal(k, 1), lambda: rotate_boxes_90(boxes)),
            (tf.equal(k, 2), lambda: rotate_boxes_180(boxes)),
            (tf.equal(k, 3), lambda: rotate_boxes_270(boxes)),
        ],
        default=lambda: boxes,
        exclusive=True
    )

    image = tf.image.random_brightness(image, max_delta=0.08)
    image = tf.image.random_contrast(image, lower=0.95, upper=1.05)

    boxes = tf.clip_by_value(boxes, 0.0, float(IMG_SIZE))
    image = tf.clip_by_value(image, 0.0, 255.0)

    return image, boxes, classes


def create_dataset_element(example, training=False):
    image = example["image"]
    boxes = tf.cast(example["boxes"], tf.float32)
    classes = tf.cast(example["classes"], tf.float32)

    image, boxes = resize_image_and_boxes(image, boxes, target_size=IMG_SIZE)

    if training:
        image, boxes, classes = augment_image_and_boxes(image, boxes, classes)

    boxes, classes = pad_or_truncate_boxes(boxes, classes)

    return image, {
        "boxes": boxes,
        "classes": classes
    }


def build_dataset(anns_list, batch_size=8, training=False):
    def generator():
        for ann in anns_list:
            image = load_geoimage(ann.filename)

            if image.ndim != 3:
                raise ValueError(f"Image invalide pour {ann.filename}: shape={image.shape}")
            if image.shape[-1] > 3:
                image = image[:, :, :3]

            boxes = np.array([obj.bb for obj in ann.objects], dtype=np.float32)
            classes = np.array([obj.category for obj in ann.objects], dtype=np.float32)

            if len(boxes) == 0:
                boxes = np.zeros((0, 4), dtype=np.float32)
                classes = np.zeros((0,), dtype=np.float32)

            yield {
                "image": image.astype(np.uint8),
                "boxes": boxes,
                "classes": classes
            }

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature={
            "image": tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
            "boxes": tf.TensorSpec(shape=(None, 4), dtype=tf.float32),
            "classes": tf.TensorSpec(shape=(None,), dtype=tf.float32),
        }
    )

    if training:
        ds = ds.shuffle(min(len(anns_list), 512), reshuffle_each_iteration=True)

    ds = ds.map(lambda x: create_dataset_element(x, training=training), num_parallel_calls=AUTO)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(AUTO)

    return ds


In [ ]:
import tensorflow as tf


def extract_data(anns_list):
    filenames = []
    tiles = []
    all_bboxes = []
    all_classes = []

    for img_ann in anns_list:
        filenames.append(img_ann.filename)
        tiles.append(list(img_ann.tile))
        all_bboxes.append([list(obj.bb) for obj in img_ann.objects])
        all_classes.append([int(obj.category) for obj in img_ann.objects])

    return filenames, tiles, all_bboxes, all_classes


filenames_train, tiles_train, bboxes_train, classes_train = extract_data(anns_train)
filenames_valid, tiles_valid, bboxes_valid, classes_valid = extract_data(anns_valid)

MAX_BOXES = min(
    max(
        max((len(b) for b in bboxes_train), default=0),
        max((len(b) for b in bboxes_valid), default=0)
    ),
    300
)

print(f"Max boxes per image: {MAX_BOXES}")

BATCH_SIZE = 12

ds_train = build_dataset(anns_train, batch_size=BATCH_SIZE, training=True)
ds_valid = build_dataset(anns_valid, batch_size=BATCH_SIZE, training=False)

print(f"Training images: {len(anns_train)}")
print(f"Validation images: {len(anns_valid)}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

for images, targets in ds_train.take(1):
    print(f"Images shape: {images.shape}")
    print(f"Boxes shape: {targets['boxes'].shape}")
    print(f"Classes shape: {targets['classes'].shape}")

    img = np.clip(images[0].numpy(), 0, 255).astype(np.uint8)
    boxes = targets["boxes"][0].numpy()
    classes = targets["classes"][0].numpy()

    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(img)

    colors = ['red', 'blue', 'green', 'orange']

    for box, cls in zip(boxes, classes):
        if cls < 0:
            continue

        xmin, ymin, xmax, ymax = box

        if xmax <= xmin or ymax <= ymin:
            continue

        w = xmax - xmin
        h = ymax - ymin
        class_id = int(cls)
        color = colors[class_id % len(colors)]

        rect = patches.Rectangle(
            (xmin, ymin),
            w,
            h,
            linewidth=2,
            edgecolor=color,
            facecolor='none'
        )
        ax.add_patch(rect)

        ax.text(
            xmin,
            max(ymin - 5, 5),
            categories[class_id],
            fontsize=8,
            color='white',
            backgroundcolor=color
        )

    valid_count = int((classes >= 0).sum())
    ax.set_title(f"Sample training image with {valid_count} objects")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    break


In [ ]:
import math
import numpy as np

print('='*60)
print('Starting training...')
print('='*60)

EPOCHS = 20

train_steps = math.ceil(len(anns_train) / BATCH_SIZE)
valid_steps = math.ceil(len(anns_valid) / BATCH_SIZE)

print(f"Epochs: {EPOCHS}")
print(f"Train steps/epoch: {train_steps}")
print(f"Valid steps/epoch: {valid_steps}")
print(f"Batch size: {BATCH_SIZE}")

history = model.fit(
    ds_train,
    validation_data=ds_valid,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

best_idx = int(np.argmin(history.history['val_loss']))
best_value = np.min(history.history['val_loss'])
print('='*60)
print(f'Best validation model: epoch {best_idx+1} - val_loss {best_value:.4f}')
print('='*60)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Valid')
axes[0].set_title('Total Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['box_loss'], label='Train')
axes[1].plot(history.history['val_box_loss'], label='Valid')
axes[1].set_title('Box Loss (CIoU)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history.history['class_loss'], label='Train')
axes[2].plot(history.history['val_class_loss'], label='Valid')
axes[2].set_title('Classification Loss (Binary crossentropy)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves_yolo.png', dpi=150)
plt.show()


In [ ]:
# Learning rate evolution (warmup + cosine decay)
import math

if 'lr' in history.history:
    lrs = history.history['lr']
else:
    lrs = [lr_schedule(e, LEARNING_RATE) for e in range(len(history.history['loss']))]

fig, ax = plt.subplots(1, figsize=(8, 4))
ax.plot(range(1, len(lrs) + 1), lrs, marker='o', color='steelblue', linewidth=2)
ax.axvspan(1, 3.5, alpha=0.1, color='orange', label='Warmup phase')
ax.axvspan(3.5, len(lrs) + 0.5, alpha=0.05, color='blue', label='Cosine decay phase')
ax.set_title('Learning Rate Schedule (Warmup + Cosine Decay)', fontsize=13)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lr_schedule.png', dpi=150)
plt.show()

In [ ]:
print("\nQuick prediction check on validation images...")

model.load_weights(CHECKPOINT_PATH)

for images, gt_bboxes in ds_valid.take(3):
    predictions = model.predict(images, verbose=0)

    img = np.clip(images[0].numpy(), 0, 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    axes[0].imshow(img)
    axes[0].set_title('Ground Truth')
    gt_boxes = gt_bboxes['boxes'][0].numpy()
    gt_classes = gt_bboxes['classes'][0].numpy()

    for box, cls in zip(gt_boxes, gt_classes):
        if cls < 0:
            continue
        xmin, ymin, xmax, ymax = box
        if xmax - xmin < 1 or ymax - ymin < 1:
            continue
        rect = patches.Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=2, edgecolor='green', facecolor='none'
        )
        axes[0].add_patch(rect)
        axes[0].text(
            xmin, max(ymin - 5, 5),
            categories[int(cls)],
            fontsize=8, color='white', backgroundcolor='green'
        )

    axes[1].imshow(img)
    axes[1].set_title('Predictions')

    num_det = int(predictions['num_detections'][0])
    pred_boxes = predictions['boxes'][0][:num_det]
    pred_classes = predictions['classes'][0][:num_det]
    pred_conf = predictions['confidence'][0][:num_det]

    print(f"Number of detections: {num_det}")

    for box, cls, conf in zip(pred_boxes, pred_classes, pred_conf):
        xmin, ymin, xmax, ymax = box
        if xmax <= xmin or ymax <= ymin:
            continue
        rect = patches.Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=2, edgecolor='red', facecolor='none'
        )
        axes[1].add_patch(rect)
        axes[1].text(
            xmin, max(ymin - 5, 5),
            f"{categories[int(cls)]} {conf:.2f}",
            fontsize=8, color='white', backgroundcolor='red'
        )

    plt.tight_layout()
    plt.show()


#### Validation
Compute validation metrics.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as col
import numpy as np
%matplotlib inline

def area_intersection(boxes, box):
    xmin = np.maximum(np.min(boxes[:, 0::2], axis=1), np.min(box[0::2]))
    ymin = np.maximum(np.min(boxes[:, 1::2], axis=1), np.min(box[1::2]))
    xmax = np.minimum(np.max(boxes[:, 0::2], axis=1), np.max(box[0::2]))
    ymax = np.minimum(np.max(boxes[:, 1::2], axis=1), np.max(box[1::2]))
    w = np.maximum(xmax - xmin + 1.0, 0.0)
    h = np.maximum(ymax - ymin + 1.0, 0.0)
    return w * h

def area_union(boxes, box):
    area_anns = (np.max(box[0::2])-np.min(box[0::2])+1.0) * (np.max(box[1::2])-np.min(box[1::2])+1.0)
    area_pred = (np.max(boxes[:, 0::2], axis=1)-np.min(boxes[:, 0::2], axis=1)+1.0) * (np.max(boxes[:, 1::2], axis=1)-np.min(boxes[:, 1::2], axis=1)+1.0)
    return area_anns + area_pred - area_intersection(boxes, box)

def calc_iou(boxes, box):
    iou = area_intersection(boxes, box) / area_union(boxes, box)
    max_value = np.max(iou)
    max_index = np.argmax(iou)
    return max_value, max_index

def calc_ap(rec, prec):
    # First append sentinel values at the end
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    # Compute the precision envelope
    for i in range(mpre.size-1, 0, -1):
        mpre[i-1] = np.maximum(mpre[i-1], mpre[i])
    # To calculate area under PR curve, look for points where X axis (recall) changes value
    i = np.where(mrec[1:] != mrec[:-1])[0]
    # and sum (\Delta recall) * prec
    ap = np.sum((mrec[i+1] - mrec[i]) * mpre[i+1])
    return ap

def draw_confusion_matrix(cm, categories):
    # Draw confusion matrix
    fig = plt.figure(figsize=[6.4*pow(len(categories), 0.5), 4.8*pow(len(categories), 0.5)])
    ax = fig.add_subplot(111)
    cm = cm.astype('float') / np.maximum(cm.sum(axis=1)[:, np.newaxis], np.finfo(np.float64).eps)
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.colormaps['Blues'] if hasattr(plt, 'colormaps') else plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]), xticklabels=categories, yticklabels=categories, ylabel='Annotation', xlabel='Prediction')
    # Rotate the tick labels and set their alignment
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    # Loop over data dimensions and create text annotations
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], '.2f'), ha="center", va="center", color="white" if cm[i, j] > thresh else "black", fontsize=int(20-pow(len(categories), 0.5)))
    fig.tight_layout()
    plt.show()

def draw_precision_recall(precisions, recalls, categories):
    # Draw precision-recall curves for each category
    fig = plt.figure(figsize=[6.4*pow(len(categories), 0.5), 4.8*pow(len(categories), 0.5)])
    ax = fig.add_subplot(111)
    plt.axis([0, 1, 0, 1])
    c_dark = list(filter(lambda x: x.startswith('dark'), col.cnames.keys()))
    aps = []
    # Compare categories for a specific algorithm
    for idx in range(len(categories)):
        plt.plot(recalls[idx], precisions[idx], color=c_dark[idx], label=categories[idx], linewidth=4.0)
        aps.append(calc_ap(recalls[idx], precisions[idx]))
    handles, labels = ax.get_legend_handles_labels()
    labels = [str(val + ' [' + "{:.3f}".format(aps[idx]) + ']') for idx, val in enumerate(labels)]
    handles = [h for (ap, h) in sorted(zip(aps, handles), key=lambda x: x[0], reverse=True)]
    labels = [l for (ap, l) in sorted(zip(aps, labels), key=lambda x: x[0], reverse=True)]
    leg = plt.legend(handles, labels, loc='upper right')
    leg.set_zorder(100)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.grid("on", linestyle="--", linewidth=2.0)
    fig.tight_layout()
    plt.show()

In [ ]:
import numpy as np
from tqdm import tqdm
import tensorflow as tf

model.load_weights(CHECKPOINT_PATH)

ds_valid_inf = build_dataset(anns_valid, batch_size=1, training=False)
iterator = iter(ds_valid_inf)

annotations, predictions = {}, {}

for ann in tqdm(anns_valid):
    annotations.setdefault(ann.filename, {})
    predictions.setdefault(ann.filename, {})

    # Scale GT boxes from original image coords to IMG_SIZE coords for fair IoU comparison
    tile_w = float(ann.tile[2]) if ann.tile[2] > 0 else float(IMG_SIZE)
    tile_h = float(ann.tile[3]) if ann.tile[3] > 0 else float(IMG_SIZE)
    scale_x = IMG_SIZE / tile_w
    scale_y = IMG_SIZE / tile_h

    for obj in ann.objects:
        cls_name = categories[obj.category]
        annotations[ann.filename].setdefault(cls_name, {'bbox': []})
        xmin, ymin, xmax, ymax = obj.bb
        scaled_bb = (xmin * scale_x, ymin * scale_y, xmax * scale_x, ymax * scale_y)
        annotations[ann.filename][cls_name]['bbox'].append(scaled_bb)

    image, _ = next(iterator)
    pred_out = model.predict(image, verbose=0)

    pred_boxes   = np.asarray(pred_out["boxes"][0])
    pred_classes = np.asarray(pred_out["classes"][0])
    pred_scores  = np.asarray(pred_out["confidence"][0])

    if "num_detections" in pred_out:
        num_det = int(np.asarray(pred_out["num_detections"][0]))
    else:
        num_det = len(pred_scores)

    for i in range(num_det):
        score = float(pred_scores[i])
        bbox  = pred_boxes[i]
        cls   = int(pred_classes[i])

        x1, y1, x2, y2 = bbox
        if score <= 0 or x2 <= x1 or y2 <= y1:
            continue

        cls_name = categories[cls]
        predictions[ann.filename].setdefault(cls_name, {'bbox': [], 'confidence': []})
        predictions[ann.filename][cls_name]['bbox'].append((float(x1), float(y1), float(x2), float(y2)))
        predictions[ann.filename][cls_name]['confidence'].append(score)


In [ ]:
threshold = 0.5
default_cls = 'BACKGROUND'
y_true, y_pred = [], []  # confusion matrix
tps, confidences = dict(), dict()  # draw precision-recall curves for each category
for cls in categories.values():
    # Compute TP, FP and FN for each image
    tps[cls], confidences[cls] = [], []
    for f in predictions:
        # Sort 'cls' predictions by confidence for each file
        pred_boxes, pred_confidences = [], []
        if cls in predictions[f].keys():
            for idx in range(len(predictions[f][cls]['bbox'])):
                pred_boxes.append(predictions[f][cls]['bbox'][idx])
                pred_confidences.append(predictions[f][cls]['confidence'][idx])
            sorted_ind = np.argsort(-np.array(pred_confidences))
            pred_boxes = np.array(pred_boxes)[sorted_ind, :]
        pred_boxes = np.array(pred_boxes).astype(float)
        # Define 'cls' annotations for each file
        anno_boxes = []
        if cls in annotations[f].keys():
            anno_boxes = annotations[f][cls]['bbox']
        anno_boxes = np.array(anno_boxes).astype(float)
        # Define horizontal or oriented bounding boxes
        anno_indices = list(range(len(anno_boxes)))
        # Compare a single prediction 'pred_box' with all annotations 'anno_boxes'
        for pred_idx, pred_box in enumerate(pred_boxes):
            # A prediction is correct if its IoU with the ground truth is above the threshold
            iou_value, ann_index = calc_iou(anno_boxes, pred_box) if len(anno_boxes) > 0 else (-1, -1)
            if iou_value > threshold and ann_index in anno_indices:
                # TP
                anno_indices.remove(int(ann_index))
                tps[cls] += [1.0]
                y_true += [cls]
            else:
                # FP
                tps[cls] += [0.0]
                y_true += [default_cls]
            y_pred += [cls]
            confidences[cls] += [pred_confidences[pred_idx]]
        # FN
        y_true += [cls] * len(anno_indices)
        y_pred += [default_cls] * len(anno_indices)
y_true, y_pred = np.array(y_true), np.array(y_pred)

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score
import numpy as np

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Compute AP metric
precision_list, recall_list, ap_list = [], [], []
for cls in categories.values():
    sorted_ind = np.argsort(-np.array(confidences[cls]))
    tp = np.cumsum(np.array(tps[cls])[sorted_ind], dtype=float)

    n_true = np.sum(y_true == cls)
    n_pred = np.sum(y_pred == cls)

    recall = np.array([0.0]) if len(tp) == 0 else tp / np.maximum(n_true, np.finfo(np.float64).eps)
    precision = np.array([0.0]) if len(tp) == 0 else tp / np.maximum(np.arange(1, len(tp) + 1), np.finfo(np.float64).eps)

    ap = calc_ap(recall, precision)
    print('> %s: Recall: %.3f%% Precision: %.3f%% AP: %.3f%%' % (cls, recall[-1]*100, precision[-1]*100, ap*100))
    precision_list.append(precision)
    recall_list.append(recall)
    ap_list.append(ap)

mean_ap = np.mean(ap_list)
print('mAccuracy: %.3f%%' % (accuracy_score(y_true, y_pred)*100))
print('mRecall: %.3f%%' % (recall_score(y_true, y_pred, average='macro', zero_division=1)*100))
print('mPrecision: %.3f%%' % (precision_score(y_true, y_pred, average='macro', zero_division=1)*100))
print('mAP: %.3f%%' % (mean_ap*100))

In [ ]:
import pandas as pd
import numpy as np

class_names_eval = ['Small car', 'Bus', 'Truck', 'Building']

final_recall = [float(r[-1]) if len(r) > 0 else 0.0 for r in recall_list]
final_precision = [float(p[-1]) if len(p) > 0 else 0.0 for p in precision_list]
final_ap = [float(ap) for ap in ap_list]

metrics_df = pd.DataFrame({
    "Class": class_names_eval,
    "Recall": final_recall,
    "Precision": final_precision,
    "AP": final_ap
})

metrics_df["Recall_%"] = metrics_df["Recall"] * 100
metrics_df["Precision_%"] = metrics_df["Precision"] * 100
metrics_df["AP_%"] = metrics_df["AP"] * 100

rare_metrics_df = metrics_df[metrics_df["Class"].isin(["Bus", "Truck"])][
    ["Class", "Recall_%", "Precision_%", "AP_%"]
].reset_index(drop=True)

print("Rare classes metrics:")
display(rare_metrics_df)

bus_row = rare_metrics_df[rare_metrics_df["Class"] == "Bus"].iloc[0]
truck_row = rare_metrics_df[rare_metrics_df["Class"] == "Truck"].iloc[0]

print(f"Bus   -> Recall: {bus_row['Recall_%']:.3f}% | AP: {bus_row['AP_%']:.3f}%")
print(f"Truck -> Recall: {truck_row['Recall_%']:.3f}% | AP: {truck_row['AP_%']:.3f}%")

In [ ]:
# Per-class AP, Recall, Precision — bar charts for report
import matplotlib.pyplot as plt
import numpy as np

class_names = list(categories.values())
colors_bar = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AP per class
bars = axes[0].bar(class_names, [ap * 100 for ap in ap_list], color=colors_bar, edgecolor='black', linewidth=0.8)
axes[0].axhline(mean_ap * 100, color='red', linestyle='--', linewidth=2, label=f'mAP = {mean_ap*100:.2f}%')
axes[0].set_title('Average Precision (AP) per Class @ IoU=0.5', fontsize=12)
axes[0].set_ylabel('AP (%)')
axes[0].set_ylim(0, max(max(ap_list) * 100 * 1.3, 5))
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)
for bar, ap in zip(bars, ap_list):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f'{ap*100:.2f}%', ha='center', va='bottom', fontsize=10)

# Recall & Precision per class
x = np.arange(len(class_names))
width = 0.35
rec_vals  = [recall_list[i][-1] * 100 for i in range(len(class_names))]
prec_vals = [precision_list[i][-1] * 100 for i in range(len(class_names))]
axes[1].bar(x - width/2, rec_vals,  width, label='Recall',    color='#42A5F5', edgecolor='black', linewidth=0.8)
axes[1].bar(x + width/2, prec_vals, width, label='Precision', color='#66BB6A', edgecolor='black', linewidth=0.8)
axes[1].set_title('Recall & Precision per Class @ IoU=0.5', fontsize=12)
axes[1].set_ylabel('%')
axes[1].set_xticks(x)
axes[1].set_xticklabels(class_names)
axes[1].set_ylim(0, 110)
axes[1].legend()
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('ap_metrics_bar.png', dpi=150)
plt.show()

print(f"\nSummary @ IoU=0.5:")
print(f"  mAP : {mean_ap*100:.2f}%")
for cls, ap, rec, prec in zip(class_names, ap_list, rec_vals, prec_vals):
    print(f"  {cls:<12} AP={ap*100:.2f}%  Recall={rec:.2f}%  Precision={prec:.2f}%")

In [ ]:
names = list(categories.values()).copy()
names.insert(0, default_cls)
cm = confusion_matrix(y_true, y_pred, labels=names)
print('Confusion matrix:')
print(cm)
draw_confusion_matrix(cm, names)
draw_precision_recall(precision_list, recall_list, list(categories.values()))


#### Testing
Try to improve the results provided in the competition.

In [ ]:
import os
import numpy as np

test_dir = None
for root, dirs, files in os.walk('.'):
    if 'xview_test' in dirs:
        test_dir = os.path.join(root, 'xview_test')
        break

if test_dir is None:
    raise FileNotFoundError("Dossier xview_test introuvable. Vérifie l'extraction du dataset.")

anns = []
for (dirpath, dirnames, filenames) in os.walk(test_dir):
    for filename in filenames:
        # Chemin relatif à partir de la racine du dataset (même convention que le train)
        rel_path = os.path.relpath(os.path.join(dirpath, filename), start='.')
        image = GenericImage(rel_path)
        image.tile = np.array([0, 0, 640, 640])
        anns.append(image)
print('Number of testing images: ' + str(len(anns)))

In [ ]:
import tensorflow as tf
import numpy as np
import json

model.load_weights("yolo_best.weights.h5")

filenames_test, tiles_test, _, _ = extract_data(anns)
ds_test = build_dataset(anns, batch_size=BATCH_SIZE, training=False)

results = []
image_idx = 0

for images, _ in ds_test:
    preds = model.predict(images, verbose=0)

    pred_boxes = np.asarray(preds["boxes"])
    pred_classes = np.asarray(preds["classes"])
    pred_scores = np.asarray(preds["confidence"])

    if "num_detections" in preds:
        num_detections = np.asarray(preds["num_detections"]).astype(int)
    else:
        num_detections = np.full(images.shape[0], pred_scores.shape[1], dtype=int)

    batch_size_actual = images.shape[0]

    for i in range(batch_size_actual):
        filename = filenames_test[image_idx]
        tile = tiles_test[image_idx]

        for box, cls, score in zip(
            pred_boxes[i][:num_detections[i]],
            pred_classes[i][:num_detections[i]],
            pred_scores[i][:num_detections[i]]
        ):
            if score <= 1e-6:
                continue

            x1, y1, x2, y2 = box
            if x2 <= x1 or y2 <= y1:
                continue

            class_id = int(cls)

            annotation_data = {
                "image_id": filename,
                "tile": [int(tile[0]), int(tile[1])],
                "category_id": class_id,
                "category_name": categories[class_id],
                "bbox": [
                    float(x1),
                    float(y1),
                    float(x2 - x1),
                    float(y2 - y1)
                ],
                "score": float(score)
            }

            results.append(annotation_data)

        image_idx += 1

print(f"Total predictions exported: {len(results)}")


In [ ]:
with open("prediction.json", "w") as outfile:
    json.dump(results, outfile)